In [8]:
from the_well.benchmark.models import UNetClassic, UNetConvNext, FNO, TFNO
from the_well.data.normalization import ZScoreNormalization
from pathlib import Path
import fsspec
import pickle
import torch
import yaml
import sys

PROJECT_ROOT = "/Users/gabrieleaccarino/Workspace/waveletflux"
sys.path.insert(0, PROJECT_ROOT)

from waveletflux.utils.well_data import get_well_dataset, get_well_inputs
from waveletflux.utils.eval import build_stats_df, build_psd_df
from waveletflux.modules.neural_ops import NeuralOperatorModule
from waveletflux.utils.wrappers import TFNOMPSSafeWrapper
from waveletflux.utils.rollout import rollout_normalized

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# Load Data

In [15]:
well_base_path = f"{PROJECT_ROOT}/data/datasets"

#well_dataset_name = 'turbulent_radiative_layer_2D'
#well_dataset_name = 'rayleigh_benard'
#well_dataset_name = 'planetswe'
well_dataset_name = 'active_matter'

stats_path = f"{well_base_path}/{well_dataset_name}/stats.yaml"

dataset = get_well_dataset(
    path=well_base_path,
    physics=well_dataset_name,
    split='test',
    steps=-1,
    transform = ['log_scalars'],   
    augment = [],
)

with fsspec.open(stats_path, mode="r") as f:
    stats = yaml.safe_load(f)

core_field_names          = dataset.core_field_names
core_constant_field_names = dataset.core_constant_field_names

normalizer = ZScoreNormalization(stats, core_field_names, core_constant_field_names)

# Load Baseline Models

In [16]:
unet_classic  = UNetClassic.from_pretrained(f"polymathic-ai/UNetClassic-{well_dataset_name}")
unet_convnext = UNetConvNext.from_pretrained(f"polymathic-ai/UNetConvNext-{well_dataset_name}")
#fno = FNO.from_pretrained(f"polymathic-ai/FNO-{well_dataset_name}")
#tfno = TFNO.from_pretrained(f"polymathic-ai/TFNO-{well_dataset_name}")
#fno           = NeuralOperatorModule.from_pretrained(f"{PROJECT_ROOT}/experiments/{well_dataset_name}/ckpts/FNO_{well_dataset_name}.pt")
#tfno          = NeuralOperatorModule.from_pretrained(f"{PROJECT_ROOT}/experiments/{well_dataset_name}/ckpts/TFNO_{well_dataset_name}.pt")

baselines = {
    'UNetClassic'   : unet_classic.to(device),
    'UNetConvNext'  : unet_convnext.to(device),
    #'FNO'           : fno.to(device),
    #'TFNO'          : TFNOMPSSafeWrapper(tfno.to(device))
}

In [17]:
N_TRAJ        = len(dataset.metadata.n_steps_per_trajectory)                # number of trajectories in the test set for this PDE 
N_STEPS_INPUT = 4                                                           # N_STEPS_INPUT sliding-window width (must match training)
ROLLOUT_STEPS = dataset.metadata.n_steps_per_trajectory[0] - N_STEPS_INPUT  # autoregressive steps after the seed window

baseline_results = {name: [] for name in baselines}
truth_list       = []   # one (T_total, H, W, C) per trajectory

with torch.no_grad():
    for traj_idx in range(N_TRAJ):
        print(f"\n{'━'*60}")
        print(f"  Trajectory {traj_idx}")
        print(f"{'━'*60}")

        item = dataset[traj_idx]
        x_full, constants = get_well_inputs(item, device=device)   # (T_total, H, W, C)
        truth_list.append(x_full.cpu())

        # Run rollout
        for model_name, model in baselines.items():
            print(f"    Running rollout for {model_name}...")
            model.eval()
            y_pred_norm, y_ref_norm, y_pred_phys, y_ref_phys = rollout_normalized(
                model, x_full, normalizer,
                n_steps_input=N_STEPS_INPUT,
                rollout_steps=ROLLOUT_STEPS,
                device=device,
            )
            baseline_results[model_name].append(y_pred_phys)

# Stack to (N_TRAJ, N_ROLLOUT_STEPS, H, W, C)
for name in baselines:
    baseline_results[name] = torch.stack(baseline_results[name], dim=0)

truth_traj = torch.stack(truth_list, dim=0)   # (N_TRAJ, T_total, H, W, C)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 0
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    Running rollout for UNetClassic...
    Running rollout for UNetConvNext...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 1
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    Running rollout for UNetClassic...
    Running rollout for UNetConvNext...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 2
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    Running rollout for UNetClassic...
    Running rollout for UNetConvNext...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 3
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    Running rollout for UNetClassic...
    Running rollout for UNetConvNext...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 4
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Save baselines to binary

In [18]:
# save baseline trajectories to disk
save_dir = Path(f'{PROJECT_ROOT}/experiments/{well_dataset_name}/results')
save_dir.mkdir(exist_ok=True)

for model_name in baselines.keys():
    save_path = save_dir / f"{model_name}_{well_dataset_name}_rollout_traj.pkl"
    with open(save_path, "wb") as f:
        pickle.dump(baseline_results[model_name], f)

# Save Ground-truth to binary

In [19]:
# save true trajectories to disk
save_dir = Path(f'{PROJECT_ROOT}/experiments/{well_dataset_name}/results')
save_dir.mkdir(exist_ok=True)

for model_name in baselines.keys():
    save_path = save_dir / f"ground_truth_{well_dataset_name}_traj.pkl"
    with open(save_path, "wb") as f:
        pickle.dump(truth_traj, f)

# Compute stats and PSD

In [20]:
meta_fields = dataset.metadata.field_names
field_names = [item for sublist in meta_fields.values() for item in sublist]

for model_name in baselines.keys():
    print(f'Computing stats for {model_name} ...')
    save_dir = Path(f'{PROJECT_ROOT}/experiments/{well_dataset_name}/metrics')

    predictions = {model_name: baseline_results[model_name]}

    stats_df = build_stats_df(
        truth        = truth_traj,  
        predictions  = predictions,
        field_names  = field_names,
        Ti           = N_STEPS_INPUT,
        lead_times   = None,
        stride       = 1,
    )

    stats_df.to_csv(save_dir / f"{model_name}_{well_dataset_name}_stats.csv", index=False)

    psd_df = build_psd_df(
        truth = truth_traj,
        predictions = predictions,
        field_names = field_names,
        Ti = N_STEPS_INPUT,
        lead_times = None,
        stride = 1,      
        min_bin_count = 3,   
    )

    psd_df.to_csv(save_dir / f"{model_name}_{well_dataset_name}_psd.csv", index=False)


Computing stats for UNetClassic ...
Computing stats for UNetConvNext ...
